# 🧠 Challenge BCI - Classification d'Imagerie Motrice

## Objectif
Construire des modèles de classification robustes pour prédire si un sujet imagine **left_hand** ou **right_hand** à partir de signaux EEG.

**Paradigme** : 6 sujets indépendants, 140 essais train (équilibrés), 60 essais test par sujet
**Métrique** : Précision moyenne sur 6 sujets
**Focus anti-overfitting** : Cross-validation stratifiée + hyperparameters conservateurs

## Plan
1. ✅ Chargement et organisation des données
2. ✅ Exploration et vérification
3. ✅ Prétraitement EEG léger
4. ✅ Extraction de caractéristiques (variance log, PSD, CSP)
5. ✅ Baseline : Variance log + LDA
6. ✅ CSP + LDA
7. ✅ Approche Riemannienne (MDM)
8. ✅ Covariance + SVM
9. ✅ Validation croisée complète
10. ✅ Sélection du meilleur modèle par sujet
11. ✅ Entraînement final et prédictions
12. ✅ Génération fichiers CSV
13. ✅ Création archive ZIP

In [78]:
import numpy as np
import pandas as pd
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Visualisation et analyse
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import butter, filtfilt
from scipy.stats import zscore

# ML et validation
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

# Pyriemann pour CSP et géométrie riemannienne
from pyriemann.estimation import Covariances
from pyriemann.classification import MDM
from pyriemann.spatialfilters import CSP

print("✅ Tous les imports réussis !")
print(f"NumPy version: {np.__version__}")

✅ Tous les imports réussis !
NumPy version: 2.4.4


## Section 1️⃣ : Chargement et organisation des données par sujet

In [79]:
# Configuration des chemins
DATA_DIR = Path('data')
OUTPUT_DIR = Path('.')

# Charger les données de tous les sujets
subjects = ['A', 'B', 'C', 'D', 'E', 'F']
data = {}

print("📂 Chargement des données...\n")
for subject in subjects:
    X_train = np.load(DATA_DIR / f'subject_{subject}_X_train.npy')
    y_train = np.load(DATA_DIR / f'subject_{subject}_y_train.npy')
    X_test = np.load(DATA_DIR / f'subject_{subject}_X_test.npy')
    
    data[subject] = {
        'X_train': X_train,
        'y_train': y_train,
        'X_test': X_test,
    }
    
    print(f"Subject {subject}:")
    print(f"  X_train: {X_train.shape} | y_train: {y_train.shape}")
    print(f"  X_test:  {X_test.shape}")

print("\n✅ Données chargées avec succès !")

📂 Chargement des données...

Subject A:
  X_train: (140, 64, 1537) | y_train: (140,)
  X_test:  (60, 64, 1537)
Subject B:
  X_train: (140, 64, 1537) | y_train: (140,)
  X_test:  (60, 64, 1537)
Subject C:
  X_train: (140, 64, 1537) | y_train: (140,)
  X_test:  (60, 64, 1537)
Subject D:
  X_train: (140, 64, 1537) | y_train: (140,)
  X_test:  (60, 64, 1537)
Subject E:
  X_train: (140, 64, 1537) | y_train: (140,)
  X_test:  (60, 64, 1537)
Subject F:
  X_train: (140, 64, 1537) | y_train: (140,)
  X_test:  (60, 64, 1537)

✅ Données chargées avec succès !


## Section 2️⃣ : Vérification des formes, classes et équilibre

In [80]:
print("🔍 Vérification des dimensions et de l'équilibre des classes\n")
print("=" * 70)

for subject in subjects:
    X_train = data[subject]['X_train']
    y_train = data[subject]['y_train']
    X_test = data[subject]['X_test']
    
    unique_classes, counts = np.unique(y_train, return_counts=True)
    
    print(f"\n📊 SUBJECT {subject}:")
    print(f"  Classes: {unique_classes}")
    print(f"  Distribution train: {dict(zip(unique_classes, counts))}")
    print(f"  Équilibre: {counts[0] == counts[1]} (50-50)")
    print(f"  Valeurs NaN: {np.isnan(X_train).sum()} | Inf: {np.isinf(X_train).sum()}")
    print(f"  Plage de valeurs: [{X_train.min():.3f}, {X_train.max():.3f}]")

print("\n✅ Toutes les vérifications passées !")

🔍 Vérification des dimensions et de l'équilibre des classes


📊 SUBJECT A:
  Classes: ['left_hand' 'right_hand']
  Distribution train: {np.str_('left_hand'): np.int64(70), np.str_('right_hand'): np.int64(70)}
  Équilibre: True (50-50)
  Valeurs NaN: 0 | Inf: 0
  Plage de valeurs: [-2101.240, 2269.859]

📊 SUBJECT B:
  Classes: ['left_hand' 'right_hand']
  Distribution train: {np.str_('left_hand'): np.int64(70), np.str_('right_hand'): np.int64(70)}
  Équilibre: True (50-50)
  Valeurs NaN: 0 | Inf: 0
  Plage de valeurs: [-2887.753, 2325.818]

📊 SUBJECT C:
  Classes: ['left_hand' 'right_hand']
  Distribution train: {np.str_('left_hand'): np.int64(70), np.str_('right_hand'): np.int64(70)}
  Équilibre: True (50-50)
  Valeurs NaN: 0 | Inf: 0
  Plage de valeurs: [-1830.334, 1730.247]

📊 SUBJECT D:
  Classes: ['left_hand' 'right_hand']
  Distribution train: {np.str_('left_hand'): np.int64(70), np.str_('right_hand'): np.int64(70)}
  Équilibre: True (50-50)
  Valeurs NaN: 0 | Inf: 0
  Plage de va

## Section 3️⃣ : Prétraitement EEG léger et reproductible

In [81]:
# Remplacement de la Section 3 (Le Retour aux Sources)
from scipy.signal import butter, sosfiltfilt

def preprocess_safe(X, fs=256, low_Hz=8, high_Hz=32):
    """
    Filtrage passe-bande strict (8-32 Hz) + Z-score.
    AUCUN recadrage temporel (on garde les 1537 points pour ne pas couper le vrai signal).
    """
    n_epochs, n_channels, n_times = X.shape
    sos = butter(4, [low_Hz, high_Hz], btype='band', fs=fs, output='sos')
    X_filt = np.zeros_like(X, dtype=np.float64)
    
    for epoch in range(n_epochs):
        for ch in range(n_channels):
            X_filt[epoch, ch, :] = sosfiltfilt(sos, X[epoch, ch, :])
            
    # Normalisation Z-score
    X_filt = X_filt - X_filt.mean(axis=2, keepdims=True)
    std = X_filt.std(axis=2, keepdims=True)
    std[std == 0] = 1e-10
    X_filt = X_filt / std
    
    return X_filt

print("🔧 Prétraitement EEG (Filtrage 8-32Hz + Z-score UNIQUEMENT)...\n")
for subject in subjects:
    data[subject]['X_train_prep'] = preprocess_safe(data[subject]['X_train'])
    data[subject]['X_test_prep'] = preprocess_safe(data[subject]['X_test'])
    print(f"Subject {subject}: ✅ filtré et prétraité (Shape conservée : {data[subject]['X_train_prep'].shape})")

🔧 Prétraitement EEG (Filtrage 8-32Hz + Z-score UNIQUEMENT)...

Subject A: ✅ filtré et prétraité (Shape conservée : (140, 64, 1537))
Subject B: ✅ filtré et prétraité (Shape conservée : (140, 64, 1537))
Subject C: ✅ filtré et prétraité (Shape conservée : (140, 64, 1537))
Subject D: ✅ filtré et prétraité (Shape conservée : (140, 64, 1537))
Subject E: ✅ filtré et prétraité (Shape conservée : (140, 64, 1537))
Subject F: ✅ filtré et prétraité (Shape conservée : (140, 64, 1537))


## Section 4️⃣ : Extraction de caractéristiques robustes

**Pourquoi les features brutes ne suffisent pas :** Les signaux EEG bruts ont 1537 points temporels par canal - impossible de classifier directement. Besoin de réduire la dimensionalité.

**Features testées :**
1. **Log-variance** : variance log de chaque canal dans le temps (simple, efficace)
2. **Power Spectral Density (PSD)** : puissance dans bandes Mu/Beta
3. **Covariance** : matrices de covariance channel-channel (géométrie riemannienne)
4. **CSP** : Common Spatial Patterns (filtres spatiaux optimisés BCI)

In [82]:
# Extracteurs de features compatibles sklearn

def temporal_variance_log(X):
    """
    Feature 1: Variance logarithmique par canal
    X shape: (n_epochs, 64, 1537) -> output: (n_epochs, 64)
    BASELINE SIMPLE MAIS TRÈS EFFICACE EN BCI
    """
    var = np.var(X, axis=-1)  # variance temporelle par channel
    var = np.nan_to_num(var, nan=1e-10, posinf=1e-10, neginf=1e-10)
    return np.log(np.clip(var, 1e-10, None))


def psd_band_features(X, fs=256):
    """
    Feature 2: PSD dans bandes Mu (8-12 Hz) et Beta (13-30 Hz)
    Extraction par FFT rapide
    X shape: (n_epochs, 64, 1537) -> output: (n_epochs, 128)
    """
    from scipy.fft import fft
    n_epochs, n_channels, n_times = X.shape
    
    # Fréquences positives
    freqs = np.fft.fftfreq(n_times, 1/fs)[:n_times//2]
    
    # Bandes d'intérêt
    mu_idx = (freqs >= 8) & (freqs <= 12)
    beta_idx = (freqs >= 13) & (freqs <= 30)
    
    features = []
    for epoch in range(n_epochs):
        psd_epoch = []
        for ch in range(n_channels):
            fft_vals = np.abs(fft(X[epoch, ch, :])[:n_times//2])**2
            psd_mu = np.log(fft_vals[mu_idx].mean() + 1e-10)
            psd_beta = np.log(fft_vals[beta_idx].mean() + 1e-10)
            psd_epoch.extend([psd_mu, psd_beta])
        features.append(psd_epoch)
    
    return np.array(features)


def covariance_features(X):
    """
    Feature 3: Vectorisation de matrices de covariance
    Plus riche que variance log mais plus dimensions
    X shape: (n_epochs, 64, 1537) -> output: (n_epochs, 64*65/2)
    """
    n_epochs, n_channels, n_times = X.shape
    cov_size = n_channels * (n_channels + 1) // 2
    
    features = np.zeros((n_epochs, cov_size))
    for epoch in range(n_epochs):
        cov_mat = np.cov(X[epoch, :, :])
        # Triangulaire supérieur + diagonale (pour symétrie)
        cov_mat = np.nan_to_num(cov_mat, nan=1e-10)
        # Remplir avec log des valeurs abs pour stabilité
        features[epoch, :] = np.log(np.abs(np.triu(cov_mat).flatten()) + 1e-10)
    
    return features

print("✅ Extracteurs de features définis (3 types)")
print("   1. Variance log (simple & efficace)")
print("   2. PSD bandes Mu/Beta (physiologiquement motivé)")
print("   3. Covariance vectorisée (riche mais complexe)")

✅ Extracteurs de features définis (3 types)
   1. Variance log (simple & efficace)
   2. PSD bandes Mu/Beta (physiologiquement motivé)
   3. Covariance vectorisée (riche mais complexe)


## Section 5️⃣ : Baseline - Variance Log + LDA

**LDA (Linear Discriminant Analysis)** : 
- Assume classes gaussiennes avec même covariance
- Régularisation "auto" (shrinkage) = robustesse
- Très stable en CV, peu de risque overfitting

In [83]:
# Model 1: Baseline - Variance Log + LDA
pipeline_baseline = Pipeline([
    ('log_var', FunctionTransformer(temporal_variance_log)),
    ('scaler', StandardScaler()),
    ('lda', LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto')),
])

print("✅ Pipeline Baseline créé: LogVar + LDA")

✅ Pipeline Baseline créé: LogVar + LDA


## Section 6️⃣ : CSP + LDA (Common Spatial Patterns)

**CSP** : Filtre spatial qui maximise rapport de variance entre classes
- Extrait 8-12 filtres spatiaux
- Très efficace en BCI motrice
- Danger overfitting si trop de composantes

In [84]:
# Remplacement de la Section 6 : CSP + LDA
from pyriemann.spatialfilters import CSP
from pyriemann.estimation import Covariances
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.pipeline import Pipeline

# Model 2: CSP + LDA (Le grand classique indétrônable en BCI)
# nfilter=4 (2 extrêmes pour chaque classe) empêche l'overfitting massif sur 140 essais
pipeline_csp = Pipeline([
    ('cov', Covariances(estimator='lwf')),
    ('csp', CSP(nfilter=4, log=True)),
    ('lda', LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto'))
])

print("✅ Pipeline CSP créé : Covariances ('lwf') + CSP (4 composantes) + LDA avec shrinkage")

✅ Pipeline CSP créé : Covariances ('lwf') + CSP (4 composantes) + LDA avec shrinkage


## Section 7️⃣ : Approche Riemannienne - MDM (Minimum Distance to Mean)

**Géométrie Riemannienne pour EEG** :
- Représenter chaque essai par matrice de covariance
- Classifier dans espace des matrices sym. positives-définies
- MDM = très simple, souvent très efficace en BCI

In [85]:
# # Model 3: Riemannian MDM (Minimum Distance to Mean)
# # Covariances + MDM classique
# pipeline_mdm = Pipeline([
#     ('cov', Covariances(estimator='ledoit_wolf')),  # Shrinkage covariance
#     ('mdm', MDM()),
# ])

# print("✅ Pipeline MDM créé: Covariances (Ledoit-Wolf) + MDM")


# Remplacement de la Section 7
from pyriemann.estimation import Covariances
from pyriemann.classification import MDM
from pyriemann.tangentspace import TangentSpace
from sklearn.linear_model import LogisticRegression

# Model 3: Riemannian MDM (Correction du bug : 'lwf' au lieu de 'ledoit_wolf')
pipeline_mdm = Pipeline([
    ('cov', Covariances(estimator='lwf')),  # Shrinkage covariance robuste
    ('mdm', MDM()),
])

# 🏆 LE MODÈLE GAGNANT (State-of-the-Art en BCI d'Imagerie Motrice)
# Projection dans l'espace tangent Riemannien + Régression Logistique très régularisée
pipeline_ts_lr = Pipeline([
    ('cov', Covariances(estimator='lwf')),
    ('ts', TangentSpace(metric='riemann')),
    ('logreg', LogisticRegression(C=0.05, penalty='l2', solver='lbfgs', max_iter=1000, random_state=42)) 
    # C=0.05 est très conservateur pour éviter l'overfitting sur 140 essais
])

print("✅ Pipelines Riemanniennes créées :")
print("   - MDM corrigé ('lwf')")
print("   - Tangent Space + Régression Logistique (Candidat Top Score)")

✅ Pipelines Riemanniennes créées :
   - MDM corrigé ('lwf')
   - Tangent Space + Régression Logistique (Candidat Top Score)


## Section 8️⃣ : SVM sur features vectorisées

**SVM avec features simples** :
- Variance log + standardisation + SVM
- C régularisé (pas très grand) pour robustesse
- Noyau linéaire par défaut (meilleure généralisation)

In [86]:
# Remplacement de la Section 8 : L'Ensemble Classifier
from sklearn.ensemble import VotingClassifier

# On combine nos deux meilleurs modèles avec un "Soft Voting" (ils votent avec leurs probabilités)
# Note : Pour que le SVM ou la LogReg fassent du soft voting, il faut qu'ils sortent des probas
pipeline_ts_lr.set_params(logreg__C=0.05) # S'assurer que la LogReg est bien régularisée

ensemble_model = VotingClassifier(
    estimators=[
        ('csp', pipeline_csp),       # Défini en Section 6
        ('riemann', pipeline_ts_lr)  # Défini en Section 7
    ],
    voting='soft'
)

# On ajoute cet ensemble à notre liste de pipelines pour la validation croisée
pipelines['ENSEMBLE (CSP + Riemann)'] = ensemble_model

print("✅ Modèle ENSEMBLE créé : Il fera voter le CSP et l'Espace Tangent ensemble !")

✅ Modèle ENSEMBLE créé : Il fera voter le CSP et l'Espace Tangent ensemble !


## Section 9️⃣ : Validation Croisée Interne par Sujet

**Stratégie anti-overfitting** :
- CV stratifiée 5-fold (respecte l'équilibre des classes)
- Évaluer sur le train AVANT d'évaluer sur le test
- Comparer variabilité entre folds (stabilité)
- Modèle qui ne surperforme pas en CV = plus confiance sur test

In [87]:
# Remplacement de la Section 9
from sklearn.model_selection import StratifiedKFold, cross_val_score

# On rassemble nos meilleurs candidats
pipelines = {
    'Baseline (LogVar+LDA)': pipeline_baseline, # Défini en Section 5
    'MDM (Riemannian)': pipeline_mdm,           # Défini en Section 7
    'TangentSpace+LR': pipeline_ts_lr,          # Défini en Section 7 (Notre candidat Top Score)
    'CSP+LDA': pipeline_csp                     # Défini en Section 6 (Le spécialiste spatial)
}

# Validation croisée pour chaque sujet
cv_results = {}
# n_splits=5 est standard et robuste pour 140 essais
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("=" * 80)
print("🔄 VALIDATION CROISÉE 5-FOLD - LA COURSE AUX PERFORMANCES")
print("=" * 80)

for subject in subjects:
    X_train = data[subject]['X_train_prep']
    y_train = data[subject]['y_train']
    
    cv_results[subject] = {}
    
    print(f"\n📊 SUBJECT {subject} (n_train={len(X_train)})")
    print("-" * 80)
    
    for pipe_name, pipeline in pipelines.items():
        try:
            scores = cross_val_score(
                pipeline, X_train, y_train,
                cv=skf, scoring='accuracy', n_jobs=-1
            )
            
            cv_results[subject][pipe_name] = {
                'mean': scores.mean(),
                'std': scores.std(),
                'scores': scores,
            }
            
            print(f"  {pipe_name:25s}: {scores.mean():.4f} ± {scores.std():.4f}")
        except Exception as e:
            print(f"  ❌ {pipe_name:25s}: ERREUR ({str(e)[:50]}...)")

print("\n✅ Validation croisée terminée ! Regarde les scores moyens.")

🔄 VALIDATION CROISÉE 5-FOLD - LA COURSE AUX PERFORMANCES

📊 SUBJECT A (n_train=140)
--------------------------------------------------------------------------------
  Baseline (LogVar+LDA)    : 0.5071 ± 0.0857
  MDM (Riemannian)         : 0.5571 ± 0.0580
  TangentSpace+LR          : 0.5000 ± 0.0452
  CSP+LDA                  : 0.5500 ± 0.0364

📊 SUBJECT B (n_train=140)
--------------------------------------------------------------------------------
  Baseline (LogVar+LDA)    : 0.5571 ± 0.0662
  MDM (Riemannian)         : 0.6786 ± 0.0958
  TangentSpace+LR          : 0.6786 ± 0.0678
  CSP+LDA                  : 0.6929 ± 0.0535

📊 SUBJECT C (n_train=140)
--------------------------------------------------------------------------------
  Baseline (LogVar+LDA)    : 0.4786 ± 0.0623
  MDM (Riemannian)         : 0.6000 ± 0.0995
  TangentSpace+LR          : 0.7000 ± 0.0623
  CSP+LDA                  : 0.7429 ± 0.0416

📊 SUBJECT D (n_train=140)
----------------------------------------------------

## Section 🔟 : Comparaison et Sélection du Meilleur Modèle

**Critères de sélection** :
1. **Score CV moyen** = performance générale
2. **Écart-type CV** = stabilité (plus faible = plus généalisable)
3. **Variance entre folds** = pas de surapprentissage excessif sur certains folds

In [88]:
# Remplacement des Sections 10 et 11 : Le Modèle BCI Sparsifié (Lasso)
from pyriemann.estimation import Covariances
from pyriemann.tangentspace import TangentSpace
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold

predictions = {}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("\n" + "=" * 80)
print("🚀 ENTRAÎNEMENT FINAL : TANGENT SPACE + SPARSE LOGISTIC REGRESSION (L1)")
print("=" * 80 + "\n")

# Pipeline de base : Covariances -> Espace Tangent -> Scaler -> LogReg L1
# Le RobustScaler est vital pour que la pénalité L1 (Lasso) s'applique équitablement
base_pipeline = Pipeline([
    ('cov', Covariances(estimator='lwf')),
    ('ts', TangentSpace(metric='riemann')),
    ('scaler', RobustScaler()),
    ('logreg', LogisticRegression(penalty='l1', solver='liblinear', random_state=42, max_iter=2000))
])

# On va laisser le modèle trouver lui-même le bon niveau de sélection de features (C) par sujet.
# Plus C est petit, plus le modèle va écraser les features inutiles à 0.
param_grid = {'logreg__C': [0.01, 0.05, 0.1, 0.3, 0.5, 1.0]}

for subject in subjects:
    X_train = data[subject]['X_train_prep']
    y_train = data[subject]['y_train']
    X_test = data[subject]['X_test_prep']
    
    print(f"Subject {subject}: Optimisation de la parcimonie (Feature Selection)...")
    
    # Recherche du meilleur C pour CE sujet spécifique
    grid = GridSearchCV(base_pipeline, param_grid, cv=skf, scoring='accuracy', n_jobs=-1)
    grid.fit(X_train, y_train)
    
    best_C = grid.best_params_['logreg__C']
    best_score = grid.best_score_
    
    # Le GridSearch a déjà réentraîné le modèle final sur tout X_train avec le meilleur C
    final_model = grid.best_estimator_
    
    # On vérifie combien de features le modèle a décidé de garder
    coefs = final_model.named_steps['logreg'].coef_[0]
    n_features_kept = np.sum(coefs != 0)
    
    print(f"  ✅ Meilleur C : {best_C} (Score CV : {best_score:.4f})")
    print(f"  🧠 Features pertinentes gardées : {n_features_kept} sur 2080 !")
    
    # Prédictions sur le test
    y_pred = final_model.predict(X_test)
    predictions[subject] = y_pred

print("\n✅ Nouvelles prédictions L1 prêtes ! Le bruit est éliminé.")


🚀 ENTRAÎNEMENT FINAL : TANGENT SPACE + SPARSE LOGISTIC REGRESSION (L1)

Subject A: Optimisation de la parcimonie (Feature Selection)...
  ✅ Meilleur C : 0.1 (Score CV : 0.5643)
  🧠 Features pertinentes gardées : 25 sur 2080 !
Subject B: Optimisation de la parcimonie (Feature Selection)...
  ✅ Meilleur C : 0.1 (Score CV : 0.6286)
  🧠 Features pertinentes gardées : 18 sur 2080 !
Subject C: Optimisation de la parcimonie (Feature Selection)...
  ✅ Meilleur C : 0.1 (Score CV : 0.8071)
  🧠 Features pertinentes gardées : 10 sur 2080 !
Subject D: Optimisation de la parcimonie (Feature Selection)...
  ✅ Meilleur C : 0.1 (Score CV : 0.7786)
  🧠 Features pertinentes gardées : 15 sur 2080 !
Subject E: Optimisation de la parcimonie (Feature Selection)...
  ✅ Meilleur C : 0.3 (Score CV : 0.5714)
  🧠 Features pertinentes gardées : 75 sur 2080 !
Subject F: Optimisation de la parcimonie (Feature Selection)...
  ✅ Meilleur C : 0.1 (Score CV : 0.5571)
  🧠 Features pertinentes gardées : 17 sur 2080 !

✅ 

## Section 1️⃣1️⃣ : Entraînement Final et Prédictions

**Processus final** :
1. Réentraîner le meilleur modèle de chaque sujet sur TOUT le train
2. Appliquer à X_test_prep pour obtenir prédictions
3. S'assurer format correct (left_hand/right_hand)

In [89]:
# predictions = {}

# print("\n" + "=" * 80)
# print("🚀 ENTRAÎNEMENT FINAL ET GÉNÉRATION PRÉDICTIONS")
# print("=" * 80 + "\n")

# for subject in subjects:
#     X_train = data[subject]['X_train_prep']
#     y_train = data[subject]['y_train']
#     X_test = data[subject]['X_test_prep']
    
#     # Récupérer le meilleur pipeline pour ce sujet
#     best_model_name = best_pipelines[subject]
#     best_pipeline = pipelines[best_model_name]
    
#     print(f"Subject {subject}:")
#     print(f"  Model: {best_model_name}")
#     print(f"  Entraînement sur {len(X_train)} échantillons...")
    
#     # Entraînement final sur tout le train
#     best_pipeline.fit(X_train, y_train)
    
#     # Prédictions sur le test
#     y_pred = best_pipeline.predict(X_test)
    
#     # Vérifications
#     print(f"  Prédictions: {len(y_pred)} | Classes: {np.unique(y_pred)}")
#     print(f"  Distribution: {dict(zip(*np.unique(y_pred, return_counts=True)))}")
    
#     predictions[subject] = y_pred
#     print(f"  ✅ Terminé\n")

# print("✅ Toutes les prédictions générées !")

## Section 1️⃣2️⃣ : Génération des fichiers de soumission CSV

**Format requis** :
- 1 fichier CSV par sujet (subject_A_y_pred.csv ... subject_F_y_pred.csv)
- 1 colonne nommée `y_pred`
- 60 lignes (tests) + 1 header
- Valeurs: `left_hand` ou `right_hand`

In [90]:
print("=" * 80)
print("📝 GÉNÉRATION FICHIERS CSV DE SOUMISSION")
print("=" * 80 + "\n")

csv_files = []

for subject in subjects:
    y_pred = predictions[subject]
    
    # Créer DataFrame
    df = pd.DataFrame({'y_pred': y_pred})
    
    # Vérifications format
    assert len(df) == 60, f"Subject {subject}: {len(df)} lignes au lieu de 60"
    assert all(df['y_pred'].isin(['left_hand', 'right_hand'])), \
        f"Subject {subject}: valeurs invalides"
    
    # Sauvegarder CSV
    filename = f'subject_{subject}_y_pred.csv'
    df.to_csv(filename, index=False)
    csv_files.append(filename)
    
    print(f"Subject {subject}:")
    print(f"  ✅ {filename} (60 lignes)")
    print(f"  Aperçu:")
    print(f"    {df.head(3)['y_pred'].tolist()}")
    print()

print("✅ Tous les fichiers CSV générés !")

📝 GÉNÉRATION FICHIERS CSV DE SOUMISSION

Subject A:
  ✅ subject_A_y_pred.csv (60 lignes)
  Aperçu:
    ['right_hand', 'left_hand', 'right_hand']

Subject B:
  ✅ subject_B_y_pred.csv (60 lignes)
  Aperçu:
    ['right_hand', 'left_hand', 'left_hand']

Subject C:
  ✅ subject_C_y_pred.csv (60 lignes)
  Aperçu:
    ['right_hand', 'left_hand', 'right_hand']

Subject D:
  ✅ subject_D_y_pred.csv (60 lignes)
  Aperçu:
    ['right_hand', 'left_hand', 'right_hand']

Subject E:
  ✅ subject_E_y_pred.csv (60 lignes)
  Aperçu:
    ['left_hand', 'right_hand', 'right_hand']

Subject F:
  ✅ subject_F_y_pred.csv (60 lignes)
  Aperçu:
    ['left_hand', 'right_hand', 'left_hand']

✅ Tous les fichiers CSV générés !


## Section 1️⃣3️⃣ : Création du fichier ZIP final

**Archive ZIP** : Rassembler tous les CSV pour soumettre à Codabench

In [91]:
import zipfile
import os

print("=" * 80)
print("🗂️ CRÉATION ARCHIVE ZIP POUR SOUMISSION")
print("=" * 80 + "\n")

# Créer ZIP
zip_filename = 'BCI_predictions.zip'
with zipfile.ZipFile(zip_filename, 'w') as zf:
    for csv_file in csv_files:
        zf.write(csv_file)
        print(f"  Ajout: {csv_file}")

print(f"\n✅ Archive créée: {zip_filename}")
print(f"   Taille: {os.path.getsize(zip_filename) / 1024:.1f} KB")
print(f"\n📤 Prêt à être soumis sur Codabench !")

🗂️ CRÉATION ARCHIVE ZIP POUR SOUMISSION

  Ajout: subject_A_y_pred.csv
  Ajout: subject_B_y_pred.csv
  Ajout: subject_C_y_pred.csv
  Ajout: subject_D_y_pred.csv
  Ajout: subject_E_y_pred.csv
  Ajout: subject_F_y_pred.csv

✅ Archive créée: BCI_predictions.zip
   Taille: 4.4 KB

📤 Prêt à être soumis sur Codabench !


---

## 📊 RÉSUMÉ DE LA SOLUTION

### 🎯 Stratégie Anti-Overfitting Implémentée

1. **Modèle par sujet** : Chaque sujet = modèle indépendant (données très spécifiques)
2. **Validation croisée 5-fold stratifiée** : Évaluation robuste avant soumission
3. **Features simples et stables** : 
   - Variance log (baseline efficace)
   - CSP (Common Spatial Patterns, classique BCI)
   - MDM Riemannien (géométrie appropriée)
   - SVM avec régularisation légère
4. **Hyperparamètres conservateurs** :
   - CSP: n_components=4 (pas trop)
   - SVM: C=0.1 (régularisation forte)
   - Shrinkage "auto" dans LDA
5. **Pas de tuning agressif** : Évite overfitting sur train
6. **Prétraitement minimal** : Centrage + normalisation (stabilité)

### 📈 Processus de Sélection

- CV 5-fold pour chaque modèle et sujet
- Sélection basée sur: score moyen + stabilité (std faible)
- Réentraînement final sur TOUT le train (pas de validation set)
- Test uniquement après validation

### 🎁 Sortie

- 6 fichiers CSV (subject_A_y_pred.csv ... subject_F_y_pred.csv)
- 1 archive ZIP prête pour Codabench
- Score final = moyenne des 6 précisions (une par sujet)